# Meko × LangChain: give your agent a memory

**Delhi Hackathon workshop.** LLMs forget everything between calls. **Meko** is a persistent context layer: memory, documents and conversations that survive restarts and can be shared between agents. You reach it through an **MCP server**, so any MCP client can use it: Claude Code and Cursor, or *your own Python code*, which is what this notebook does.

You will:
1. Connect to the Meko MCP server from LangChain
2. Create a **datapack** and a **conversation**
3. Do full **CRUD on memories** (add, search, read, update, delete)
4. **Promote** a memory and upload a document into the **knowledge base**
5. Build a LangChain **agent** that decides on its own when to remember and recall

### Four ideas you need

| Concept | What it is |
|---|---|
| **Datapack** | An isolated workspace (think "one database per project or agent"). Everything below lives inside one. |
| **Memory** | Short facts an agent writes ("Ravi is allergic to peanuts"). Searched by meaning, not keywords. |
| **Knowledge base (KB)** | The datapack's trusted layer: uploaded documents plus memories an owner **promoted**. |
| **Conversation** | A stored chat thread. Memories and messages are tagged with it, so you can trace who said what. |

```
your code ──LangChain──► MCP client ──HTTPS──► mcp.mekodata.ai ──► datapack { memories | knowledge base | conversations }
```

## 1. Setup

**Meko API key:** sign in at [cloud.mekodata.ai](https://cloud.mekodata.ai) → **Settings → API Keys → Generate API key**. Copy it right away (it starts with `mko_tkn_` and is shown only once).

**Pick an LLM.** It is only used by the agent in section 7; sections 2 to 6 call Meko directly with no LLM. Choose one in the next cell:

| Option | `LLM` value | What you need |
|---|---|---|
| Local, free (default) | `"ollama:llama3.2"` | [Install Ollama](https://ollama.com/download), then run `ollama pull llama3.2` |
| Other local models | `"ollama:qwen3:8b"`, `"ollama:mistral-small3.2"` | `ollama pull <model>`; it must support tool calling |
| OpenAI | `"openai:gpt-4.1-mini"` | `OPENAI_API_KEY` |

Fill in the config cell below. Leave a key empty and the notebook asks for it when it runs (handy when sharing your screen). Don't commit a notebook with your keys pasted in.

In [ ]:
%pip install -q langchain-mcp-adapters langchain langchain-ollama langchain-openai httpx

In [ ]:
# ---- Config: everything this notebook needs ----
MEKO_API_KEY = ""                 # mko_tkn_...  (empty = ask when run)
LLM = "ollama:llama3.2"           # local and free
# LLM = "openai:gpt-4.1-mini"     # needs OPENAI_API_KEY below
OPENAI_API_KEY = ""               # only for openai: models (empty = ask when run)

MEKO_URL = "https://mcp.mekodata.ai/mcp"
AGENT_ID = "delhi-hack:mentor-bot"  # labels who wrote each memory; use "<team>:<app>"

import os
from getpass import getpass

MEKO_API_KEY = MEKO_API_KEY or getpass("MEKO_API_KEY: ")
if LLM.startswith("openai:"):
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY or getpass("OPENAI_API_KEY: ")

## 2. Connect to the Meko MCP server

`MultiServerMCPClient` from **langchain-mcp-adapters** speaks MCP and turns every server tool into a normal LangChain tool. Meko authenticates with `Authorization: Bearer <key>` and also requires a `User-Agent` header.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "meko": {
        "transport": "streamable_http",
        "url": MEKO_URL,
        "headers": {
            "Authorization": f"Bearer {MEKO_API_KEY}",
            "User-Agent": "delhi-hackathon-notebook/1.0",
        },
    }
})

tools = await client.get_tools()
meko = {t.name: t for t in tools if not t.name.startswith("amp_")}  # amp_* are infra tools, not used here
print(f"{len(meko)} Meko tools:", ", ".join(sorted(meko)))

Each tool is a LangChain `BaseTool`, so you can call it yourself with `.ainvoke({...})` or hand it to an agent. Meko replies with JSON text, so this small helper calls a tool and returns a Python dict.

In [ ]:
import asyncio, base64, hashlib, json, uuid
import httpx

async def call(tool: str, **args):
    """Call a Meko MCP tool and return its JSON reply as Python data."""
    for attempt in range(3):  # retry brief server hiccups (e.g. a 502)
        try:
            blocks = await meko[tool].ainvoke(args)
            break
        except Exception:
            if attempt == 2:
                raise
            await asyncio.sleep(3)
    text = "".join(b.get("text", "") for b in blocks if b.get("type") == "text") if isinstance(blocks, list) else blocks
    data = json.loads(text)
    if isinstance(data, dict) and set(data) == {"result"}:  # some replies arrive wrapped once more
        data = json.loads(data["result"])
    if isinstance(data, dict) and data.get("error"):
        raise RuntimeError(f"{tool}: {data['error']} {data.get('detail', '')}")
    return data

def show(data, limit=1200):
    print(json.dumps(data, indent=2, default=str)[:limit])

## 3. Create a datapack

Datapack names are **unique across all of Meko**, so we add a random suffix. Most tools take a `conversation_id` for tracing; we have no conversation yet, so a random hex id is fine for these setup calls.

In [ ]:
setup_id = uuid.uuid4().hex

dp = await call("datapack_create", name=f"delhi-hack-{setup_id[:6]}", conversation_id=setup_id)
DATAPACK_ID = dp["datapack_id"]
show(dp)

In [ ]:
# Update: add a description, then read the datapack back
await call("datapack_update", datapack_id=DATAPACK_ID, description="Workshop: mentor bot memory", conversation_id=setup_id)

info = await call("datapack_describe", datapack_id=DATAPACK_ID, conversation_id=setup_id)
show({k: info[k] for k in ["datapack_name", "description", "grant", "memory_count", "knowledge_count"]})

In [ ]:
# List the datapacks your key can see (filtered by name)
for d in await call("datapack_list", query="delhi-hack", conversation_id=setup_id):
    print(d["datapack_name"], "→", d["datapack_id"])

## 4. Create a conversation and log a turn

A conversation is the thread every later call belongs to. Store chat turns with `conversation_add_message`; reuse `CONV_ID` for everything that follows.

Meko also **extracts memories from stored turns** on its own, so you'll see an extra memory about team Byte appear in section 5.

In [ ]:
conv = await call("conversation_create", agent_id=AGENT_ID, datapack_id=DATAPACK_ID, title="Mentor desk, day 1")
CONV_ID = conv["id"]

await call("conversation_add_message", conversation_id=CONV_ID, agent_id=AGENT_ID, datapack_id=DATAPACK_ID,
           input="Hi! We're team Byte, building a campus lost-and-found app.",
           output="Great idea! What stack are you using?")

thread = await call("conversation_get", conversation_id=CONV_ID, agent_id=AGENT_ID, datapack_id=DATAPACK_ID, include_messages=True)
print(thread["title"], "| messages:", thread["message_count"])
for m in thread["messages"]:
    print("user:", m["input"], "\nbot: ", m["output"])

## 5. Memory CRUD

The arguments are the same every time, so we keep them in `ctx`.

In [ ]:
ctx = dict(conversation_id=CONV_ID, agent_id=AGENT_ID, datapack_id=DATAPACK_ID)

facts = [
    "Team Byte is building a campus lost-and-found app in Python with FastAPI.",
    "Ravi from team Byte is allergic to peanuts.",
    "Team Byte's demo slot is Saturday at 4pm in Hall B.",
]
ids = []
for fact in facts:
    res = await call("memory_add", text=fact, **ctx)
    ids.append(res["results"][0]["id"])
    print(ids[-1], "←", fact)

### Read: search by meaning
The query below shares almost no words with the stored fact. Meko combines vector similarity with keyword matching, so it still finds it.

In [ ]:
hits = await call("memory_search", query="what can't Ravi eat?", limit=3, **ctx)
for h in hits["results"]:
    print(f"{h['score']:.2f}  {h['memory']}")

In [ ]:
# Read: everything in the datapack (about 20 most recent) or one memory by id
everything = await call("memory_get_all", **ctx)
for m in everything["memories"]:
    print(m["id"], m["data"])

memory_id = hits["results"][0]["id"]
show(await call("memory_get_by_id", memory_id=memory_id, **ctx), limit=300)

In [ ]:
# Update: replace a memory's text
await call("memory_update", memory_id=ids[2], text="Team Byte's demo moved to Saturday 5pm in Hall C.", **ctx)
print((await call("memory_get_by_id", memory_id=ids[2], **ctx))["memory"])

In [ ]:
# Delete: remove one memory
await call("memory_delete_by_id", memory_id=ids[2], **ctx)
print([m["data"] for m in (await call("memory_get_all", **ctx))["memories"]])

## 6. Knowledge base: promote memories and upload documents

Working memory is noisy: anything an agent writes lands there. The **knowledge base** holds what is *trusted*:

- **Promote** moves a memory into the KB. It is one-way and only the datapack's owner or maintainers may do it. The memory leaves working memory, so `memory_search` no longer returns it; `knowledgebase_search` does.
- **Upload** a document (PDF, Markdown, TXT, CSV, JSON, HTML). Meko chunks and indexes it for search.

In [ ]:
# Promote the tech-stack fact (ids[0] from section 5)
show(await call("memory_promote", memory_ids=[ids[0]], **ctx))

for m in (await call("memory_get_all", promoted=True, **ctx))["promoted_memories"]:
    print("KB ←", m["data"])

### Upload a document
Upload is two steps: `knowledgebase_upload_document` returns a pre-signed URL, then you POST the file to it. Meko indexes it in the background.

In [ ]:
faq = b"""# Delhi Hackathon FAQ
- Submissions close Sunday at 10am. Late entries are not judged.
- Lunch is served at 1pm in the atrium; dinner at 8pm.
- Wi-Fi network: HACK-DEL. The password is printed on your badge.
- Mentors are at the help desk near Hall A from 10am to 10pm.
"""

up = await call("knowledgebase_upload_document", filename="faq.md", size=len(faq),
                content_hash=hashlib.sha256(faq).hexdigest(),
                content_head_base64=base64.b64encode(faq[:3072]).decode(),
                conversation_id=CONV_ID, datapack_id=DATAPACK_ID)
r = httpx.post(up["url"], data=up["fields"], files={"file": ("faq.md", faq)})
print("upload HTTP", r.status_code)

In [ ]:
# Indexing usually takes a few seconds
for _ in range(12):
    hits = await call("knowledgebase_search", query="when do submissions close?", limit=3,
                      conversation_id=CONV_ID, datapack_id=DATAPACK_ID)
    if any("Submissions" in h["chunk_text"] for h in hits["results"]):
        break
    await asyncio.sleep(5)

for h in hits["results"]:
    print("•", h["chunk_text"][:120].replace("\n", " "))

Notice the promoted memory and the FAQ are both in the KB now.

### `context_search`: one call across everything
This is the best default read for an agent: it searches working memory, the KB and past conversations together, and drops weak matches. It is strict, so for documents and promoted facts `knowledgebase_search` is the more forgiving read.

In [ ]:
res = (await call("context_search", query="What can't Ravi eat?", limit=3, **ctx))["results"]
for source in ["memory", "knowledge_base", "conversation"]:
    print(source, "→", json.dumps(res.get(source), default=str)[:150])

## 7. A LangChain agent that uses Meko on its own

Everything so far was us calling tools by hand. Now an LLM agent decides when to **save** and when to **recall**.

You *could* hand it the raw MCP tools (`meko["memory_add"]` etc.), but then the LLM has to copy ids into every call and pick the right search. A sturdier pattern is two small LangChain tools that wrap the MCP calls: ids stay in code, and `recall` always reads both working memory and the KB.

Memory gets written two ways:
- **The agent calls `remember`**: instant, but small models (like Llama 3.2 3B) often forget to.
- **Our code logs every turn** with `conversation_add_message`, and Meko extracts the lasting facts into memory in the background (about 30 s). This works with any model.

Real-world example: a **hackathon mentor bot** that remembers each team across sessions.

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

@tool
async def remember(fact: str) -> str:
    """Save one lasting fact about a user or their team (name, role, preferences, constraints)."""
    res = await call("memory_add", text=fact, **ctx)
    return "saved" if res["results"] else "already known"

@tool
async def recall(query: str) -> str:
    """Search long-term memory and the knowledge base (event FAQ, verified team facts)."""
    mem = await call("memory_search", query=query, limit=5, **ctx)
    kb = await call("knowledgebase_search", query=query, limit=3, conversation_id=CONV_ID, datapack_id=DATAPACK_ID)
    found = [f"[memory] {r['memory']}" for r in mem["results"]] + [f"[kb] {r['chunk_text']}" for r in kb["results"]]
    return "\n".join(found) or "nothing found"

SYSTEM = """You are a friendly mentor bot at the Delhi Hackathon with long-term memory.
For every user message:
1. If the user shares something lasting about themselves or their team, call remember once per fact.
2. Call recall with a short search query.
3. Answer in 1-3 sentences, using only what recall returned or the user just said."""

def new_agent():
    return create_agent(init_chat_model(LLM, temperature=0), [remember, recall], system_prompt=SYSTEM)

async def ask(agent, question):
    out = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})
    for m in out["messages"]:
        for tc in getattr(m, "tool_calls", None) or []:
            print(f"  🔧 {tc['name']}({next(iter(tc['args'].values()), '')!r})")
    answer = out["messages"][-1].content
    print("🤖", answer)
    # Log the turn; Meko extracts lasting facts from it in the background
    await call("conversation_add_message", input=question, output=answer, **ctx)

In [ ]:
agent = new_agent()
await ask(agent, "Hey, I'm Priya from team Byte. I handle the frontend and I prefer React. When do submissions close?")

Now simulate a **brand-new session**: a fresh agent with no chat history. Everything it knows has to come from Meko. First, give Meko a moment to extract facts from the logged turn.

In [ ]:
for _ in range(18):
    if any("Priya" in m["data"] for m in (await call("memory_get_all", **ctx))["memories"]):
        break
    await asyncio.sleep(5)
print("Meko now knows:", [m["data"] for m in (await call("memory_get_all", **ctx))["memories"] if "Priya" in m["data"]])

In [ ]:
fresh_agent = new_agent()
await ask(fresh_agent, "I'm on team Byte. Remind me: what's our backend stack, who does frontend, and is there anything Ravi can't eat?")

In [ ]:
# Everything in working memory now
for m in (await call("memory_get_all", **ctx))["memories"]:
    print("•", m["data"])

## 8. Clean up

Deleting a datapack is permanent: it removes all its memories, KB documents and conversations. Set `DELETE_DATAPACK = True` when you're done (or keep it and look around in the [Meko portal](https://cloud.mekodata.ai)).

In [ ]:
DELETE_DATAPACK = False

if DELETE_DATAPACK:
    show(await call("datapack_delete", datapack_id=DATAPACK_ID, conversation_id=setup_id))
else:
    print("Kept datapack", DATAPACK_ID)

## Cheat sheet: Meko MCP tools

| Area | Tools |
|---|---|
| Datapacks | `datapack_create`, `datapack_list`, `datapack_search`, `datapack_describe`, `datapack_update`, `datapack_delete` |
| Conversations | `conversation_create`, `conversation_add_message`, `conversation_get`, `conversation_list`, `conversation_update`, `conversation_delete`, `track_token_usage` |
| Memory | `memory_add`, `memory_search`, `memory_get_all`, `memory_get_by_id`, `memory_update`, `memory_delete_by_id`, `memory_delete_all`, `memory_promote` |
| Knowledge base | `knowledgebase_upload_document`, `knowledgebase_document_status`, `knowledgebase_search`, `knowledgebase_delete_document` |
| Search everything | `context_search` (pass `force_source="knowledge_base"` to read only the KB) |
| Files | `artifact_put`, `artifact_get` |

**Gotchas**
- Datapack names are global: always add a unique prefix or suffix.
- Every request needs a `User-Agent` header; `X-Api-Key` style headers don't authenticate.
- Promoted memories leave working memory: read them with `knowledgebase_search`, not `memory_search`.
- Wrap MCP calls in your own tools (like `remember` / `recall`) to pin ids in code instead of trusting the LLM to copy them.

**Next steps:** connect the same datapack to Claude Code or Cursor so your coding agent and your app share memory. Docs: [docs.mekodata.ai](https://docs.mekodata.ai).